# Unitree Go1：複数速度で補正モデルを学習し、3速度で比較する

**Colab ProでL4を選び、「すべてのセルを実行」してください。Drive接続の承認が必要です。**
旧ノートブックの実験結果は上書きしません。コードはファイルごとに読みやすいセルで内蔵しています。

出発点は公開済みのWalk These Ways Go1方策です。元の歩行方策と履歴適応モジュールを固定し、
0.10〜0.30m/sの速度指令に対応する**補正モデルを1つ**学習します。
速度ごとに別モデルを作る構成ではありません。補正モデルも速度指令を観測として受け取ります。

学習後、同じ補正モデルを0.10・0.20・0.30m/sで評価し、「公開方策のみ」と比較します。
以前の2m先で減速する課題とは異なり、今回は一定速度指令で約60秒歩き続けます。
公開方策の実測速度と指令値には差があり得るため、両方と速度RMSEを記録します。

## 1 実験設定
学習時の速度はエピソードごとに0.10〜0.30m/sから一様に抽選し、そのエピソードでは一定です。
速度の途中変更への追従は今回の対象に含めません。

揺れを4段階で増やし、各300万、合計1200万ステップを学習します。
学習seedは700、評価seedは1000〜1004で、以前の比較の300〜304とは分けます。
評価用seedで条件やモデルを選び直しません。最終段階の1つのモデルで全速度を比較します。

前回L4では300万ステップの学習に約24分かかりました。ただし今回の環境での速度は未実測です。
全4段階の学習に加え、揺れる甲板の30エピソードと静止甲板の30エピソードを評価します。
**合計3〜5時間程度の実験枠を確保してください。これは今回の通し実行の実測値ではありません。**
静止甲板の対照評価を省くならFLAT_CONTROL=Falseにできますが、揺れへの改善と物理移植誤差の補償を区別しにくくなります。

In [ ]:
from pathlib import Path

OUTPUT = '/content/drive/MyDrive/ship_robot_runs/experiment04_multispeed'
ENVS = 32
FLAT_CONTROL = True

CONFIG = {
    'output': OUTPUT,
    'assets': OUTPUT + '/public_assets',
    'envs': ENVS,
    'speed_min': 0.10,
    'speed_max': 0.30,
    'eval_speeds': [0.10, 0.20, 0.30],
    'conditions': [[2.0, 1.4, 0.02], [6.0, 4.0, 0.04],
                   [12.0, 8.0, 0.07], [18.0, 12.0, 0.10]],
    'steps_per_level': 3_000_000,
    'train_seed': 700,
    'test_seeds': [1000, 1001, 1002, 1003, 1004],
    'seconds': 60.0,
    'flat_control': FLAT_CONTROL,
}
assert ENVS in [8, 16, 32, 64, 128]
assert 0 < CONFIG['speed_min'] < CONFIG['speed_max'] <= 0.30
assert CONFIG['conditions'] and CONFIG['steps_per_level'] > 0
assert all(CONFIG['speed_min'] <= s <= CONFIG['speed_max'] for s in CONFIG['eval_speeds'])
print(CONFIG)

## 2 Driveを接続
以前の補正policy.pklは読み込みません。課題が異なるため、公開歩行方策に加える補正を新しく学習します。
途中で切れた場合は、今回の実験の補正だけを再開に使います。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
Path(OUTPUT).mkdir(parents=True, exist_ok=True)

## 3 コードをファイルごとに保存
各`%%writefile`セルは、その下のコードを保存します。この段階では学習を開始しません。
通常は編集不要です。各見出しから、確認したいファイルのセルを読めます。

### 3.1 保存先を準備

In [ ]:
import os
import sys
import json
import subprocess
import hashlib
import shutil
import signal
import queue
import threading
import time

WORK = Path('/content/unitree_go1_multispeed')
WORK.mkdir(parents=True, exist_ok=True)
archive = Path(OUTPUT) / 'source_snapshot'
archive.mkdir(parents=True, exist_ok=True)
print('コード保存先:', WORK)

### 3.2 `model.py`

Go1と40mの甲板、関節対応、公開方策の観測定義。

In [ ]:
%%writefile /content/unitree_go1_multispeed/model.py
"""Pinned Menagerie Go1, torque PD as in WTW hardware deployment."""
from pathlib import Path
import xml.etree.ElementTree as ET
import numpy as np
import mujoco

LEGS = ['FL','FR','RL','RR']
NAMES = [f'{leg}_{axis}_joint' for leg in LEGS for axis in ['hip','thigh','calf']]
HOME = np.array([.1,.8,-1.5,-.1,.8,-1.5,.1,1.,-1.5,-.1,1.,-1.5],np.float32)
SCALE = np.tile([.125,.25,.25],4).astype(np.float32)

def build(asset_dir, out):
    src = Path(asset_dir)/'unitree_go1'
    root = ET.parse(src/'go1.xml').getroot()
    root.find('compiler').set('meshdir',str((src/'assets').resolve()))
    opt=root.find('option'); opt.attrib.update(timestep='.0025',cone='pyramidal',solver='Newton',iterations='10',ls_iterations='10',impratio='1')
    # WTW hardware applies explicit 20/0.5 PD. Remove Menagerie position servos
    # and extra joint damping, which otherwise would apply PD twice.
    for joint in root.findall('.//default//joint'):
        joint.attrib.update(damping='0',frictionloss='0',armature='.01')
    for geom in root.findall('.//default//geom'):
        if geom.get('type')=='cylinder': geom.set('type','capsule')
        if geom.get('condim'): geom.attrib.update(condim='3',solimp='.9 .95 .001',solref='.02 1')
    for geom in root.findall('.//worldbody//geom'):
        if geom.get('type')=='cylinder': geom.set('type','capsule')
        if geom.get('class')!='visual': geom.attrib.update(contype='1',conaffinity='2')
    world=root.find('worldbody')
    robot=world.find("body[@name='trunk']"); robot.set('pos','-12 0 1.24')
    light=world.find('light'); light.attrib.update(mode='fixed',pos='0 -2 5'); light.attrib.pop('target',None)
    ET.SubElement(world,'geom',type='plane',size='10 10 .1',rgba='.1 .23 .32 1',contype='0',conaffinity='0')
    heave=ET.Element('body',name='heave',pos='0 0 .8')
    ET.SubElement(heave,'joint',name='heave',type='slide',axis='0 0 1',damping='0')
    ET.SubElement(heave,'inertial',mass='1',pos='0 0 0',diaginertia='1 1 1')
    roll=ET.SubElement(heave,'body',name='roll')
    ET.SubElement(roll,'joint',name='roll',axis='1 0 0',damping='0')
    ET.SubElement(roll,'inertial',mass='1',pos='0 0 0',diaginertia='1 1 1')
    deck=ET.SubElement(roll,'body',name='deck')
    ET.SubElement(deck,'joint',name='pitch',axis='0 1 0',damping='0')
    ET.SubElement(deck,'geom',name='deck',type='box',size='20 2 .1',mass='100',contype='2',conaffinity='1',condim='3',friction='1 .005 .0001',rgba='.35 .45 .5 1')
    for x in np.arange(-19.5,20.,.5):
        ET.SubElement(deck,'geom',type='box',pos=f'{x} 0 .102',size='.008 1.9 .002',mass='0',contype='0',conaffinity='0',rgba='.8 .8 .7 1')
    world.insert(1,heave)
    root.remove(root.find('actuator')); act=ET.SubElement(root,'actuator')
    for n in ['heave','roll','pitch']: ET.SubElement(act,'motor',name=n,joint=n)
    for n in NAMES:
        lim=35.55 if 'calf' in n else 23.7
        ET.SubElement(act,'motor',name=n,joint=n,ctrlrange=f'{-lim} {lim}')
    key=root.find('keyframe')
    if key is not None: root.remove(key)
    visual=root.find('visual')
    if visual is None: visual=ET.SubElement(root,'visual')
    ET.SubElement(visual,'global',offwidth='960',offheight='540')
    ET.SubElement(visual,'headlight',ambient='.5 .5 .5',diffuse='.8 .8 .8',specular='.2 .2 .2')
    out=Path(out);out.parent.mkdir(parents=True,exist_ok=True)
    ET.ElementTree(root).write(out,encoding='unicode')
    return out

def mapping(m):
    # Menagerie order is FR, FL, RR, RL; network order is FL, FR, RL, RR.
    ids=np.array([m.joint(n).id for n in NAMES])
    return m.jnt_qposadr[ids],m.jnt_dofadr[ids]

def observation(q,v,rotation,action,previous,phase,speed,xp=np,side=0.,yaw=0.):
    command=xp.array([speed*2,side*2,yaw*.25,0,3,0.5,0,0,0.5,0.08*.15,0,0,.25,.4,0])
    clocks=xp.sin(2*xp.pi*(phase+xp.array([.5,0,0,.5])))
    return xp.clip(xp.concatenate([rotation.T@xp.array([0.,0.,-1.]),command,
        q-xp.asarray(HOME),v*.05,action,previous,clocks]),-100,100)

### 3.3 `policy.py`

元のGo1方策の推論。重みを固定します。

In [ ]:
%%writefile /content/unitree_go1_multispeed/policy.py
"""ELU inference, unchanged WTW adaptation module + actor weights."""
import numpy as np
import jax.numpy as jp
import jax
class FrozenPolicy:
    def __init__(self,path):
        with np.load(path) as z:self.w={k:jp.array(z[k]) for k in z.files}
    def net(self,x,prefix,layers):
        for i in layers:
            x=x@self.w[f'{prefix}.{i}.weight'].T+self.w[f'{prefix}.{i}.bias']
            if i!=layers[-1]:x=jax.nn.elu(x)
        return x
    def __call__(self,h):
        latent=self.net(h,'adaptation_module',[0,2,4])
        return self.net(jp.concatenate([h,latent],axis=-1),'actor_body',[0,2,4,6])

### 3.4 `ship_env.py`

速度抽選、速度指令の保持、甲板の物理駆動、速度追従の報酬。

In [ ]:
%%writefile /content/unitree_go1_multispeed/ship_env.py
"""WTW frozen Go1 gait + learned joint-target residual, physical moving deck."""
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import mujoco
from mujoco import mjx
import jax
import jax.numpy as jp
from flax import struct
from brax.envs.base import Env,State
from model import build,mapping,HOME,SCALE,observation
from policy import FrozenPolicy

@dataclass(frozen=True)
class Config:
    roll_deg:float=2.
    pitch_deg:float=1.4
    heave_m:float=.02
    speed:float=.1
    random_speed:bool=False
    random_start:bool=False
    speed_min:float=.10
    speed_max:float=.30
    sim_dt:float=.0025
    substeps:int=8

@struct.dataclass
class Pipeline:
    data:object
    waves:object
    history:object
    last:object
    previous:object
    residual:object
    speed:object

def motion(t,waves,amplitude):
    s=jp.clip(t/5.,0,1); e=10*s**3-15*s**4+6*s**5
    ed=jp.where(t<5,(30*s**2-60*s**3+30*s**4)/5,0.)
    angle=waves[...,0]*t+waves[...,1]
    raw=jp.mean(jp.sin(angle),axis=-1);rd=jp.mean(waves[...,0]*jp.cos(angle),axis=-1)
    return amplitude*e*raw,amplitude*(ed*raw+e*rd)

class ShipEnv(Env):
    def __init__(self,config=Config(),assets='assets'):
        if not np.isclose(config.sim_dt*config.substeps,.02):
            raise ValueError('WTW policy requires a 20 ms control period')
        self.config=config; self.assets=Path(assets).resolve()
        self.xml=build(self.assets,self.assets/'ship_scene.xml')
        self.mj_model=mujoco.MjModel.from_xml_path(str(self.xml))
        self.mj_model.opt.timestep=config.sim_dt
        self.sys=mjx.put_model(self.mj_model)
        self.qi,self.vi=map(jp.array,mapping(self.mj_model))
        self.base_id=self.mj_model.body('trunk').id;self.deck_id=self.mj_model.body('deck').id
        self.foot_ids=jp.array([self.mj_model.geom(n).id for n in ['FL','FR','RL','RR']])
        self.q0=jp.array(self.mj_model.qpos0).at[self.qi].set(HOME)
        # Reset pose is always the same apart from a free-base x translation.
        # Precompute its full native forward state once. The next MJX step
        # recomputes all kinematics; reset observations need only these poses.
        native=mujoco.MjData(self.mj_model)
        native.qpos[:]=np.asarray(self.q0)
        mujoco.mj_forward(self.mj_model,native)
        self._data=mjx.put_data(self.mj_model,native)
        robot=self.mj_model.body_rootid==self.base_id
        self.body_robot=jp.array(robot[:,None])
        self.geom_robot=jp.array(robot[self.mj_model.geom_bodyid,None])
        self.frozen=FrozenPolicy(self.assets/'policy.npz')
        self.amplitude=jp.array([config.heave_m,np.deg2rad(config.roll_deg),np.deg2rad(config.pitch_deg)])
        self.robot_mass=float(self.mj_model.body_subtreemass[self.base_id])
        self.limits=jp.array(self.mj_model.actuator_ctrlrange[3:,1])
    @property
    def dt(self):return self.config.sim_dt*self.config.substeps
    @property
    def action_size(self):return 12
    @property
    def observation_size(self):return 94
    @property
    def backend(self):return 'mjx'
    def random_waves(self,rng):
        k1,k2=jax.random.split(rng)
        periods=jax.random.uniform(k1,(3,3),minval=2.,maxval=6.)
        phases=jax.random.uniform(k2,(3,3),minval=-jp.pi,maxval=jp.pi)
        return jp.stack([2*jp.pi/periods,phases],axis=-1)
    def relative(self,d):
        rot=d.xmat[self.deck_id];delta=d.xpos[self.base_id]-d.xpos[self.deck_id]
        roll=d.qpos[1];omega=jp.array([d.qvel[1],jp.cos(roll)*d.qvel[2],jp.sin(roll)*d.qvel[2]])
        deck_v=jp.array([0.,0.,d.qvel[0]])+jp.cross(omega,delta)
        return rot.T@delta,rot.T@(d.qvel[3:6]-deck_v)
    def baseobs(self,d,last,previous,speed):
        pos,_=self.relative(d); rot=d.xmat[self.base_id]
        yaw=jp.arctan2(rot[1,0],rot[0,0])
        return observation(d.qpos[self.qi],d.qvel[self.vi],rot,last,previous,
                           jp.mod(d.time*3.,1.),speed,jp,
                           jp.clip(-.4*pos[1],-.2,.2),jp.clip(-2*yaw,-.5,.5))
    def observe(self,p):
        d=p.data;_,vel=self.relative(d)
        up=d.xmat[self.base_id].T@d.xmat[self.deck_id][:,2]
        return jp.concatenate([p.history[-70:],vel,up,d.qpos[:3],d.qvel[:3],p.residual])
    def reset(self,rng):
        kw,kv,kx=jax.random.split(rng,3)
        q0=self.q0.at[3].set(jax.random.uniform(kx,(),minval=-12.,maxval=10.)) if self.config.random_start else self.q0
        shift=jp.array([q0[3]-self.q0[3],0.,0.])
        d=self._data.replace(qpos=q0,
            xpos=self._data.xpos+self.body_robot*shift,
            geom_xpos=self._data.geom_xpos+self.geom_robot*shift)
        speed=jax.random.uniform(kv,(),minval=self.config.speed_min,maxval=self.config.speed_max) if self.config.random_speed else jp.asarray(self.config.speed,jp.float32)
        zero=jp.zeros(12);obs=self.baseobs(d,zero,zero,speed)
        h=jp.concatenate([jp.zeros(2030),obs])
        p=Pipeline(d,self.random_waves(kw),h,zero,zero,zero,speed)
        z=jp.array(0.,jp.float32)
        metrics={n:z for n in ['velocity','height','slip_proxy','reward']}
        return State(p,self.observe(p),z,z,metrics,{})
    def control(self,d,waves,qtarget):
        target,tv=motion(d.time,waves,self.amplitude)
        deck=jp.array([30000.,40000.,2666667.])*(target-d.qpos[:3])+jp.array([1600.,4000.,266667.])*(tv-d.qvel[:3])+d.qfrc_bias[:3]
        deck=deck.at[0].add(self.robot_mass*9.81)
        tau=jp.clip(20*(qtarget-d.qpos[self.qi])-.5*d.qvel[self.vi],-self.limits,self.limits)
        return jp.concatenate([deck,tau])
    def feet_relative(self,d):return (d.geom_xpos[self.foot_ids]-d.xpos[self.deck_id])@d.xmat[self.deck_id]
    def step(self,state,action):
        old=state.pipeline_state; action=jp.clip(action,-1,1)
        base=jp.clip(self.frozen(old.history),-10,10)
        # Residual is bounded in joint-target radians; base policy never changes.
        total=jp.clip(base+.12*action/jp.array(SCALE),-10,10)
        qtarget=jp.array(HOME)+jp.array(SCALE)*total
        def substep(d,_):return mjx.step(self.sys,d.replace(ctrl=self.control(d,old.waves,qtarget))),None
        d,_=jax.lax.scan(substep,old.data,None,length=self.config.substeps)
        d=mjx.forward(self.sys,d)
        obs=self.baseobs(d,total,old.last,old.speed);h=jp.concatenate([old.history[70:],obs])
        p=Pipeline(d,old.waves,h,total,old.last,action,old.speed)
        pos,vel=self.relative(d);upright=d.xmat[self.base_id][2,2]
        feet=self.feet_relative(d);oldfeet=self.feet_relative(old.data)
        near=(feet[:,2]<.14)&(oldfeet[:,2]<.14)
        slip=jp.mean(near*jp.sum(((feet-oldfeet)/self.dt)[:,:2]**2,axis=1))
        track=jp.exp(-((vel[0]-old.speed)**2+vel[1]**2)/.01)
        reward=(2*track+upright-10*(pos[2]-.37)**2-.1*jp.sum(d.qvel[6:9]**2)
                -.3*slip-.15*pos[1]**2-.0002*jp.sum(d.ctrl[3:]**2)
                -.02*jp.sum(action**2)-.02*jp.sum((action-old.residual)**2))
        bad=(pos[2]<.26)|(upright<.45)|(jp.abs(pos[0])>19.5)|(jp.abs(pos[1])>1.5)|~jp.all(jp.isfinite(d.qpos))|~jp.all(jp.isfinite(d.qvel))
        done=bad.astype(jp.float32);reward=jp.nan_to_num(reward,nan=-10.)-5*done
        metrics={'velocity':vel[0],'height':pos[2]-.1,'slip_proxy':slip,'reward':reward}
        return state.replace(pipeline_state=p,obs=self.observe(p),reward=reward,done=done,metrics=metrics)
    def write_xml(self,path):Path(path).write_text(self.xml.read_text(),encoding='utf-8')

### 3.5 `reset_wrapper.py`

各エピソード終了時に、速度・揺れ・開始位置を抽選し直す処理。

In [ ]:
%%writefile /content/unitree_go1_multispeed/reset_wrapper.py
"""Resample commands and waves at EVERY episode boundary.

Brax's stock AutoResetWrapper restores the first state, so it would repeat
the initial per-environment command. This wrapper draws a new reset key.
"""
import jax
import jax.numpy as jp
from brax.envs.base import Wrapper
from brax.envs.wrappers.training import VmapWrapper,EpisodeWrapper

class ResamplingAutoReset(Wrapper):
    def reset(self,rng):
        split=jax.vmap(lambda k:jax.random.split(k,2))(rng)
        state=self.env.reset(split[:,0])
        state.info['reset_rng']=split[:,1]
        return state
    def step(self,state,action):
        state.info['steps']=jp.where(state.done,0,state.info['steps'])
        state=state.replace(done=jp.zeros_like(state.done))
        state=self.env.step(state,action)
        split=jax.vmap(lambda k:jax.random.split(k,2))(state.info['reset_rng'])
        def fresh(_):
            reset=self.env.reset(split[:,0])
            return reset.pipeline_state,reset.obs
        new_pipeline,new_obs=jax.lax.cond(jp.any(state.done),fresh,
            lambda _:(state.pipeline_state,state.obs),operand=None)
        def select(new,old):
            mask=state.done.reshape(state.done.shape+(1,)*(old.ndim-state.done.ndim))
            return jp.where(mask,new,old)
        pipeline=jax.tree.map(select,new_pipeline,state.pipeline_state)
        obs=jax.tree.map(select,new_obs,state.obs)
        state.info['reset_rng']=jp.where(state.done[:,None],split[:,1],state.info['reset_rng'])
        return state.replace(pipeline_state=pipeline,obs=obs)

def wrap(env,episode_length=1000,action_repeat=1,randomization_fn=None):
    if randomization_fn is not None:raise ValueError('This task randomizes within reset, not model parameters')
    return ResamplingAutoReset(EpisodeWrapper(VmapWrapper(env),episode_length,action_repeat))

### 3.6 `train.py`

1つの補正モデルを複数速度で学習。

In [ ]:
%%writefile /content/unitree_go1_multispeed/train.py
"""PPO curriculum runner. Checkpoints are trusted local Python pickle files."""
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')
import argparse
from dataclasses import asdict
from functools import partial
import json
import importlib.metadata
import pickle
from pathlib import Path
import time
import jax
import jax.numpy as jp
from brax.training.agents.ppo import networks, train as ppo
from ship_env import Config, ShipEnv
from reset_wrapper import wrap as training_wrap

def NETWORK(observation_size, action_size, preprocess_observations_fn=lambda x, _: x, **kwargs):
    # Early running variances can be tiny. Bound normalized inputs to avoid
    # extreme critic predictions during the first few updates.
    def preprocess(obs, params):
        return jp.clip(preprocess_observations_fn(obs, params), -10., 10.)
    return networks.make_ppo_networks(observation_size, action_size,
        preprocess_observations_fn=preprocess,
        policy_hidden_layer_sizes=(128,128,128),
        value_hidden_layer_sizes=(128,128,128), **kwargs)


def load(path):
    # Never load a pickle obtained from an untrusted source.
    with open(path, 'rb') as f:
        return pickle.load(f)


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--stage', choices=['wave'], default='wave')
    p.add_argument('--assets', default='assets')
    p.add_argument('--speed-min', type=float, default=.1)
    p.add_argument('--speed-max', type=float, default=.3)
    p.add_argument('--steps', type=int, default=2_000_000)
    p.add_argument('--envs', type=int, default=64)
    p.add_argument('--out', default='runs/multispeed')
    p.add_argument('--restore')
    p.add_argument('--seed', type=int, default=0)
    p.add_argument('--roll', type=float, default=1.0)
    p.add_argument('--pitch', type=float, default=.7)
    p.add_argument('--heave', type=float, default=.01)
    p.add_argument('--speed', type=float, default=.1)
    p.add_argument('--allow-cpu', action='store_true')
    p.add_argument('--tiny', action='store_true', help='API smoke test, not training')
    p.add_argument('--evals', type=int, default=9)
    a = p.parse_args()
    if not a.allow_cpu and jax.default_backend() != 'gpu':
        raise RuntimeError('GPU is not active. Select Colab L4, restart runtime, check jax.devices().')
    if not a.tiny and (a.envs > 256 or 256 % a.envs):
        p.error('--envs must divide 256: use 8, 16, 32, 64, 128 or 256')
    cfg = Config(roll_deg=a.roll if a.stage == 'wave' else 0.,
                 pitch_deg=a.pitch if a.stage == 'wave' else 0.,
                 heave_m=a.heave if a.stage == 'wave' else 0.,
                 speed=a.speed, random_speed=True, random_start=True, speed_min=a.speed_min, speed_max=a.speed_max)
    if not 0 < a.speed_min < a.speed_max:
        p.error("Require 0 < speed-min < speed-max")
    env = ShipEnv(cfg, a.assets)
    out = Path(a.out)
    out.mkdir(parents=True, exist_ok=True)
    env.write_xml(out/'scene.xml')
    start = time.monotonic()
    print('Devices:', jax.devices(), flush=True)
    print(f'Episode speed commands: uniform [{a.speed_min:.2f}, {a.speed_max:.2f}] m/s; resampled each episode', flush=True)
    print('Configuration:', asdict(cfg), flush=True)
    (out/'config.json').write_text(json.dumps(asdict(cfg), indent=2))
    (out/'versions.json').write_text(json.dumps({name: importlib.metadata.version(name)
        for name in ['jax', 'jaxlib', 'mujoco', 'mujoco-mjx', 'brax',
                     'flax', 'optax', 'orbax-checkpoint', 'numpy', 'scipy']}, indent=2))

    actual_step = 0
    def save(step, make_policy, params):
        nonlocal actual_step
        actual_step = int(step)
        payload = {'params': jax.device_get(params), 'config': asdict(cfg),
                   'steps_this_run': int(step), 'seed': a.seed,
                   'network': [128, 128, 128], 'task': 'constant-speed-multispeed-v1'}
        tmp = out/'policy.tmp'
        with tmp.open('wb') as f:
            pickle.dump(payload, f)
        tmp.replace(out/'policy.pkl')
        print(f'policy.pkl saved: {int(step):,} steps', flush=True)

    def progress(step, metrics):
        row = {'step': int(step), 'elapsed_s': time.monotonic()-start}
        row.update({k: float(v) for k, v in metrics.items()})
        with (out/'progress.jsonl').open('a') as f:
            f.write(json.dumps(row)+'\n')
        print(json.dumps(row), flush=True)

    previous = load(a.restore) if a.restore else None
    if previous and previous.get('task') != 'constant-speed-multispeed-v1':
        raise ValueError('Use only checkpoints from this new task; old goal-stop residuals are not interchangeable.')
    old = previous['params'] if previous else None
    _, params, _ = ppo.train(
        environment=env, num_timesteps=8 if a.tiny else a.steps,
        num_envs=2 if a.tiny else a.envs, episode_length=10 if a.tiny else 1000,
        action_repeat=1, learning_rate=3e-4, entropy_cost=.01,
        discounting=.99, unroll_length=2 if a.tiny else 16,
        batch_size=2 if a.tiny else 64, num_minibatches=1 if a.tiny else 4,
        num_updates_per_batch=1 if a.tiny else 4,
        normalize_observations=True, network_factory=NETWORK, wrap_env_fn=training_wrap,
        seed=a.seed, num_evals=2 if a.tiny else a.evals,
        num_eval_envs=2 if a.tiny else 8, deterministic_eval=True,
        progress_fn=progress, policy_params_fn=save,
        restore_params=old, restore_value_fn=True,
        run_evals=True,
    )
    save(actual_step, None, params)
    print('Finished. Training reward alone does not establish successful walking.', flush=True)


if __name__ == '__main__':
    main()

### 3.7 `evaluate_speeds.py`

一定速度での評価。速度を変えても同じコンパイル済み処理と方策を使用。

In [ ]:
%%writefile /content/unitree_go1_multispeed/evaluate_speeds.py
"""Paired constant-speed tests, no episode resets, shared compiled functions."""
import os
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE','false')
import argparse,json,time
from dataclasses import asdict
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jp
from brax.training.acme import running_statistics
from brax.training.agents.ppo import networks
from ship_env import ShipEnv,Config,motion,Pipeline
from train import NETWORK,load

def evaluate(assets,out,condition,speeds,seeds,seconds=60,policy_path=None,allow_cpu=False):
    if not allow_cpu and jax.default_backend()!='gpu':raise RuntimeError('Select GPU runtime')
    cfg=Config(roll_deg=condition[0],pitch_deg=condition[1],heave_m=condition[2])
    env=ShipEnv(cfg,assets);out=Path(out);out.mkdir(parents=True,exist_ok=True)
    def reset(rng,speed):
        state=env.reset(rng);p=state.pipeline_state
        obs=env.baseobs(p.data,p.last,p.previous,speed)
        p=p.replace(speed=speed,history=jp.concatenate([jp.zeros(2030),obs]))
        return state.replace(pipeline_state=p,obs=env.observe(p))
    reset=jax.jit(reset);step=jax.jit(env.step);relative=jax.jit(env.relative)
    targets=jax.jit(lambda d,w:motion(d.time,w,env.amplitude)[0])
    inference=None
    if policy_path:
        payload=load(policy_path)
        if payload.get('task')!='constant-speed-multispeed-v1':raise ValueError('Wrong checkpoint task')
        net=NETWORK(94,12,preprocess_observations_fn=running_statistics.normalize)
        inference=jax.jit(networks.make_inference_fn(net)(payload['params'],deterministic=True))
    results=[]
    print('Devices:',jax.devices(),'condition:',condition,flush=True)
    for speed in speeds:
        for seed in seeds:
            dest=out/f'speed_{speed:.2f}'/f'seed_{seed}';dest.mkdir(parents=True,exist_ok=True)
            env.write_xml(dest/'scene.xml')
            print(f'speed={speed:.2f}, seed={seed}: reset/compile',flush=True)
            state=reset(jax.random.PRNGKey(seed),jp.asarray(speed,jp.float32));jax.block_until_ready(state.obs)
            start_x=float(relative(state.pipeline_state.data)[0][0])
            rows=[];qs=[];vs=[];ts=[];reason='completed';last=time.monotonic()
            for i in range(round(seconds/env.dt)):
                a=inference(state.obs,jax.random.PRNGKey(seed+i))[0] if inference else jp.zeros(12)
                state=step(state,a);p=state.pipeline_state;d=p.data
                pos,vel=map(np.asarray,relative(d));q=np.asarray(d.qpos);v=np.asarray(d.qvel)
                t=float(d.time);rot=np.asarray(d.xmat[env.base_id]);target=np.asarray(targets(d,p.waves))
                finite=np.isfinite(q).all() and np.isfinite(v).all() and np.isfinite(float(state.reward))
                if not finite:reason='numerical_error'
                elif pos[2]<.26 or rot[2,2]<.45:reason='fall_criterion'
                elif abs(pos[0])>19.5 or abs(pos[1])>1.5:reason='out_of_bounds'
                elif bool(state.done):reason='other_failure'
                roll=np.arctan2(rot[2,1],rot[2,2]);pitch=np.arcsin(np.clip(-rot[2,0],-1,1))
                rows.append([t,*pos,*vel,float(state.metrics['slip_proxy']),roll,pitch,*q[:3],*target])
                if (i+1)%2==0 or reason!='completed':qs.append(q);vs.append(v);ts.append(t)
                if i==0 or time.monotonic()-last>15:
                    print(f'speed={speed:.2f}, seed={seed}: sim {t:.1f}/{seconds:.0f}s, x={pos[0]:.2f}',flush=True);last=time.monotonic()
                if reason!='completed':break
            r=np.asarray(rows);active=r[r[:,0]>=5.]
            use=active if len(active) else r
            summary={'seed':seed,'speed_m_s':speed,'requested_seconds':seconds,'simulated_seconds':float(r[-1,0]),
                'failed':reason!='completed','termination_reason':reason,'fall_detected':reason=='fall_criterion',
                'completed':reason=='completed','success':reason=='completed',
                'forward_displacement_m':float(r[-1,1]-start_x),'start_x_m':start_x,
                'mean_forward_speed_m_s':float(np.mean(use[:,4])),
                'speed_rmse_m_s':float(np.sqrt(np.mean((use[:,4]-speed)**2))),
                'mean_slip_proxy_m2_s2':float(np.mean(use[:,7])),
                'body_roll_rms_deg':float(np.rad2deg(np.sqrt(np.mean(use[:,8]**2)))),
                'body_pitch_rms_deg':float(np.rad2deg(np.sqrt(np.mean(use[:,9]**2)))),
                'body_roll_std_deg':float(np.rad2deg(np.std(use[:,8]))),
                'body_pitch_std_deg':float(np.rad2deg(np.std(use[:,9]))),
                'deck_tracking_rmse':np.sqrt(np.mean((r[:,10:13]-r[:,13:16])**2,axis=0)).tolist(),
                'metric_window':'t >= 5 s; before failure only; fallback whole trace if duration < 5 s',
                'metric_window_seconds':float(len(use)*env.dt),
                'policy':str(policy_path) if policy_path else 'WTW pretrained only',
                'config':{**asdict(cfg),'speed':speed},'no_episode_resets':True}
            if not all(np.isfinite(summary[k]) for k in ['speed_rmse_m_s','mean_slip_proxy_m2_s2','body_roll_rms_deg']):summary['termination_reason']='numerical_error';summary['failed']=True;summary['completed']=False;summary['success']=False
            (dest/'summary.json').write_text(json.dumps(summary,indent=2))
            np.savetxt(dest/'metrics.csv',r,delimiter=',',header='time,x,y,z,vx,vy,vz,slip_proxy,body_roll,body_pitch,heave,roll,pitch,target_heave,target_roll,target_pitch',comments='')
            np.savez_compressed(dest/'trajectory.npz',qpos=qs,qvel=vs,time=ts)
            results.append(summary);(out/'results.json').write_text(json.dumps(results,indent=2))
            print(json.dumps(summary),flush=True)
            if summary['termination_reason']=='numerical_error':raise RuntimeError('Nonfinite result; stop instead of counting as a fall')
    return results

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--assets',required=True);p.add_argument('--out',required=True)
    p.add_argument('--condition',nargs=3,type=float,required=True);p.add_argument('--speeds',nargs='+',type=float,default=[.1,.2,.3])
    p.add_argument('--seeds',nargs='+',type=int,required=True);p.add_argument('--seconds',type=float,default=60)
    p.add_argument('--policy');p.add_argument('--allow-cpu',action='store_true');a=p.parse_args()
    if a.seconds<=0 or any(s<=0 or s>.3 for s in a.speeds):p.error('Require positive seconds and speeds <= .30')
    evaluate(a.assets,a.out,a.condition,a.speeds,a.seeds,a.seconds,a.policy,a.allow_cpu)

### 3.8 `report.py`

速度別集計と、同じseedの左右差を保存。

In [ ]:
%%writefile /content/unitree_go1_multispeed/report.py
"""Aggregate paired runs without rewarding early termination."""
import csv,json,statistics
from pathlib import Path
METRICS=['mean_forward_speed_m_s','speed_rmse_m_s','mean_slip_proxy_m2_s2',
         'body_roll_rms_deg','body_pitch_rms_deg','body_roll_std_deg','body_pitch_std_deg',
         'forward_displacement_m']
def summarize(baseline,trained,out,condition):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    keys=lambda r:(r['speed_m_s'],r['seed'])
    assert sorted(map(keys,baseline))==sorted(map(keys,trained))
    grouped=[]
    for speed in sorted(set(r['speed_m_s'] for r in baseline)):
        for label,rows in [('baseline',baseline),('trained',trained)]:
            runs=[r for r in rows if r['speed_m_s']==speed];complete=[r for r in runs if r['completed']]
            result={'speed_m_s':speed,'controller':label,'count':len(runs),
                'completed_count':len(complete),'fall_count':sum(r['fall_detected'] for r in runs),
                'out_of_bounds_count':sum(r['termination_reason']=='out_of_bounds' for r in runs),
                'completed_only_metric_count':len(complete)}
            # Do not let short failed traces appear better in aggregate means.
            for k in METRICS:
                result[k]=statistics.mean(r[k] for r in complete) if complete else None
            grouped.append(result)
    paired=[]
    bb={keys(r):r for r in baseline};tt={keys(r):r for r in trained}
    for key in sorted(bb):
        b,t=bb[key],tt[key]
        if 'config' in b and 'config' in t:assert b['config']==t['config'],'Paired conditions differ'
        if 'requested_seconds' in b and 'requested_seconds' in t:assert b['requested_seconds']==t['requested_seconds'],'Paired durations differ'
        pair={'speed_m_s':key[0],'seed':key[1],'both_completed':b['completed'] and t['completed'],
              'baseline_reason':b['termination_reason'],'trained_reason':t['termination_reason']}
        for k in METRICS:pair[k+'_trained_minus_baseline']=t[k]-b[k] if pair['both_completed'] else None
        paired.append(pair)
    report={'condition':condition,'baseline':baseline,'trained':trained,'speed_summary':grouped,
        'paired_differences':paired,'aggregation':'continuous metric means use completed runs only; paired differences use jointly completed seeds only; inspect completion counts first',
        'video_seed':min(r['seed'] for r in baseline),'video_selection':'first predefined test seed, regardless of outcome',
        'task':'constant-speed-multispeed-v1'}
    (out/'final_report.json').write_text(json.dumps(report,indent=2))
    for name,rows in [('speed_summary.csv',grouped),('paired_differences.csv',paired)]:
        with (out/name).open('w',newline='') as f:
            writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    print(json.dumps(grouped,indent=2),flush=True)
    return report

### 3.9 `run_experiment.py`

4段階学習、最終評価、再実行時のスキップを管理。

In [ ]:
%%writefile /content/unitree_go1_multispeed/run_experiment.py
"""Fixed curriculum; one residual checkpoint for every test speed."""
import argparse,hashlib,json,os,queue,signal,subprocess,sys,threading,time
from pathlib import Path
from report import summarize
ROOT=Path(__file__).resolve().parent
def digest(path):return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def run(args,log):
    log=Path(log);log.parent.mkdir(parents=True,exist_ok=True)
    print('RUN:',*args,flush=True)
    p=subprocess.Popen([sys.executable,'-u',*map(str,args)],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=os.name!='nt')
    q=queue.Queue()
    def reader():
        for line in p.stdout:q.put(line)
        q.put(None)
    threading.Thread(target=reader,daemon=True).start();start=time.monotonic()
    try:
        with log.open('a',encoding='utf-8') as f:
            while True:
                try:line=q.get(timeout=30)
                except queue.Empty:line=f'[alive] elapsed {(time.monotonic()-start)/60:.1f} min\n'
                if line is None:break
                print(line,end='',flush=True);f.write(line);f.flush()
        if p.wait():raise RuntimeError(f'Child failed. See {log}')
    finally:
        if p.poll() is None:
            if os.name=='nt':p.terminate()
            else:os.killpg(p.pid,signal.SIGTERM)
            try:p.wait(timeout=10)
            except subprocess.TimeoutExpired:
                if os.name=='nt':p.kill()
                else:os.killpg(p.pid,signal.SIGKILL)
                p.wait()
def main():
    parser=argparse.ArgumentParser();parser.add_argument('--config',required=True);a=parser.parse_args()
    cfg=json.loads(Path(a.config).read_text());out=Path(cfg['output']);out.mkdir(parents=True,exist_ok=True)
    saved=out/'experiment_config.json'
    if saved.exists() and json.loads(saved.read_text())!=cfg:raise RuntimeError('Config changed; use a new OUTPUT')
    saved.write_text(json.dumps(cfg,indent=2))
    signature={'config':cfg,'source':{p.name:digest(p) for p in ROOT.glob('*.py')},
               'public_policy':digest(Path(cfg['assets'])/'policy.npz')}
    model=None
    for i,condition in enumerate(cfg['conditions'],1):
        print(f'\n=== level_{i:02d}: roll/pitch/heave={condition}; train speed {cfg["speed_min"]:.2f}..{cfg["speed_max"]:.2f} m/s ===',flush=True)
        dest=out/f'level_{i:02d}';dest.mkdir(parents=True,exist_ok=True)
        marker=dest/'completed.json';policy=dest/'policy.pkl'
        expected={'signature':signature,'level':i,'restore':digest(model) if model else None}
        if marker.exists():
            data=json.loads(marker.read_text())
            if data['expected']!=expected or data['policy_sha256']!=digest(policy):raise RuntimeError('Saved stage differs; choose a new OUTPUT')
            print('SKIP completed training:',dest,flush=True)
        else:
            restore=policy if policy.exists() else model
            args=['train.py','--assets',cfg['assets'],'--stage','wave','--roll',condition[0],
                '--pitch',condition[1],'--heave',condition[2],'--steps',cfg['steps_per_level'],
                '--envs',cfg['envs'],'--seed',cfg['train_seed'],
                '--speed-min',cfg['speed_min'],'--speed-max',cfg['speed_max'],
                '--evals',9,'--out',dest]
            if restore:args+=['--restore',restore]
            run(args,dest/'console.log')
            marker.write_text(json.dumps({'expected':expected,'policy_sha256':digest(policy)},indent=2))
        model=policy
    condition=cfg['conditions'][-1]
    all_results=[]
    for label,policy in [('baseline',None),('trained',model)]:
        dest=out/('heldout_'+label);marker=dest/'completed.json'
        expected={'signature':signature,'policy_sha256':digest(policy) if policy else None,'condition':condition}
        if marker.exists() and json.loads(marker.read_text())==expected:
            print('SKIP completed evaluation:',label,flush=True)
        else:
            args=['evaluate_speeds.py','--assets',cfg['assets'],'--out',dest,'--condition',*condition,
                  '--speeds',*cfg['eval_speeds'],'--seeds',*cfg['test_seeds'],'--seconds',cfg['seconds']]
            if policy:args+=['--policy',policy]
            run(args,dest/'console.log');marker.write_text(json.dumps(expected,indent=2))
        all_results.append(json.loads((dest/'results.json').read_text()))
    summarize(*all_results,out,condition)
    if cfg.get('flat_control',True):
        controls=[]
        for label,policy in [('baseline',None),('trained',model)]:
            dest=out/('flat_control_'+label);marker=dest/'completed.json'
            expected={'signature':signature,'policy_sha256':digest(policy) if policy else None,'condition':[0,0,0]}
            if not marker.exists() or json.loads(marker.read_text())!=expected:
                args=['evaluate_speeds.py','--assets',cfg['assets'],'--out',dest,'--condition',0,0,0,
                      '--speeds',*cfg['eval_speeds'],'--seeds',*cfg['test_seeds'],'--seconds',cfg['seconds']]
                if policy:args+=['--policy',policy]
                run(args,dest/'console.log');marker.write_text(json.dumps(expected,indent=2))
            controls.append(json.loads((dest/'results.json').read_text()))
        summarize(*controls,out/'flat_control_report',[0,0,0])
    run(['render_speeds.py','--root',out,'--seed',cfg['test_seeds'][0],'--speeds',*cfg['eval_speeds'],
         '--clip-seconds',20],out/'render_console.log')
    print('All done:',out/'videos/speed_comparison_fixed.mp4',flush=True)
if __name__=='__main__':
    if os.name!='nt':signal.signal(signal.SIGTERM,lambda *_:(_ for _ in ()).throw(KeyboardInterrupt()))
    main()

### 3.10 `render_speeds.py`

世界固定カメラで各速度20秒、計60秒の左右比較動画を生成。

In [ ]:
%%writefile /content/unitree_go1_multispeed/render_speeds.py
"""20-second fixed-world clips per speed; no JAX/FFmpeg fork conflict."""
import os
os.environ.setdefault('MUJOCO_GL','glfw' if os.name=='nt' else 'egl')
os.environ['IMAGEIO_FFMPEG_NO_PREVENT_SIGINT']='1'
import argparse,json
from pathlib import Path
import numpy as np
import mujoco
import imageio.v2 as imageio
from PIL import Image,ImageDraw,ImageFont
def main():
    p=argparse.ArgumentParser();p.add_argument('--root',required=True);p.add_argument('--seed',type=int,required=True)
    p.add_argument('--speeds',nargs='+',type=float,default=[.1,.2,.3]);p.add_argument('--clip-seconds',type=float,default=20)
    a=p.parse_args();root=Path(a.root);dest=root/'videos';dest.mkdir(parents=True,exist_ok=True)
    fontpath=Path('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf')
    if not fontpath.exists():fontpath=Path('C:/Windows/Fonts/arial.ttf')
    font=ImageFont.truetype(str(fontpath),22) if fontpath.exists() else ImageFont.load_default()
    combined=imageio.get_writer(dest/'speed_comparison_fixed.mp4',fps=25,codec='libx264',macro_block_size=2)
    try:
        for speed in a.speeds:
            folders=[root/('heldout_'+label)/f'speed_{speed:.2f}'/f'seed_{a.seed}' for label in ['baseline','trained']]
            summaries=[json.loads((f/'summary.json').read_text()) for f in folders]
            assert summaries[0]['config']==summaries[1]['config']
            tracks=[np.load(f/'trajectory.npz') for f in folders]
            m=mujoco.MjModel.from_xml_path(str(folders[0]/'scene.xml'));data=[mujoco.MjData(m),mujoco.MjData(m)]
            m.vis.global_.offwidth=1280;m.vis.global_.offheight=720
            camera=mujoco.MjvCamera();camera.lookat[:]=[-12+(speed+.10)*a.clip_seconds/2,0,1.]
            camera.distance=6+(speed+.10)*a.clip_seconds;camera.azimuth=90;camera.elevation=-18
            renderer=mujoco.Renderer(m,width=1280,height=720)
            writer=imageio.get_writer(dest/f'speed_{speed:.2f}_fixed.mp4',fps=25,codec='libx264',macro_block_size=2)
            print('Render fixed-camera clip:',speed,flush=True)
            try:
                for i in range(round(a.clip_seconds*25)):
                    panels=[];t=(i+1)/25
                    for k in range(2):
                        tr=tracks[k];idx=max(0,min(np.searchsorted(tr['time'],t,side='right')-1,len(tr['time'])-1))
                        data[k].qpos[:]=tr['qpos'][idx];data[k].qvel[:]=tr['qvel'][idx];mujoco.mj_forward(m,data[k])
                        renderer.update_scene(data[k],camera=camera);im=Image.fromarray(renderer.render());draw=ImageDraw.Draw(im)
                        label=['WTW pretrained only','WTW + multispeed residual'][k]
                        frozen=summaries[k]['failed'] and t>summaries[k]['simulated_seconds']+.04
                        status='FROZEN: '+summaries[k]['termination_reason'] if frozen else 'running'
                        draw.rectangle((0,0,1280,82),fill=(15,20,30))
                        draw.text((12,6),f'{label} | command {speed:.2f} m/s | seed {a.seed}',font=font,fill='white')
                        draw.text((12,41),f'clip t={t:.2f}s | fixed world camera | {status}',font=font,fill='white')
                        panels.append(np.asarray(im))
                    frame=np.concatenate(panels,axis=1);writer.append_data(frame);combined.append_data(frame)
                    if i%250==0:print(f'{i+1}/{round(a.clip_seconds*25)} frames',flush=True)
            finally:
                writer.close();renderer.close()
                for tr in tracks:tr.close()
    finally:combined.close()
    (dest/'video_details.json').write_text(json.dumps({'seed':a.seed,'speeds':a.speeds,'seconds_per_speed':a.clip_seconds,
        'camera':'fixed for each speed; identical left/right; position differs across speed clips',
        'evaluation_duration':'60 seconds normally; video uses first 20 seconds per speed',
        'failure_padding':'final pose frozen and labelled'},indent=2))
if __name__=='__main__':main()

### 3.11 `fetch_assets.py`

コミットとSHA-256を固定した公開資産の取得。

In [ ]:
%%writefile /content/unitree_go1_multispeed/fetch_assets.py
"""Download pinned public files; validate every SHA-256 before conversion."""
import argparse,hashlib,json,urllib.request
from pathlib import Path
def main():
    p=argparse.ArgumentParser();p.add_argument('--assets',default='assets');a=p.parse_args()
    root=Path(a.assets);root.mkdir(parents=True,exist_ok=True)
    manifest=json.loads(Path(__file__).with_name('assets_manifest.json').read_text())
    for row in manifest['files']:
        dest=root/row['path'];dest.parent.mkdir(parents=True,exist_ok=True)
        if not dest.exists() or hashlib.sha256(dest.read_bytes()).hexdigest()!=row['sha256']:
            print('Download:',row['path'],flush=True)
            request=urllib.request.Request(row['url'],headers={'User-Agent':'Go1-ship-notebook/1.0'})
            data=urllib.request.urlopen(request,timeout=120).read()
            if hashlib.sha256(data).hexdigest()!=row['sha256']:raise RuntimeError('SHA256 mismatch: '+row['path'])
            temp=dest.with_suffix(dest.suffix+'.tmp');temp.write_bytes(data);temp.replace(dest)
        print('SHA256 OK:',row['path'],flush=True)
    (root/'provenance.json').write_text(json.dumps(manifest,indent=2))
if __name__=='__main__':main()

### 3.12 `convert.py`

公開重みの変換と設定・数値出力の互換性検査。

In [ ]:
%%writefile /content/unitree_go1_multispeed/convert.py
"""Convert only pinned upstream weights, and compare PyTorch with JAX."""
import argparse, json, pickle, io
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import jax
import jax.numpy as jp

def main():
    p=argparse.ArgumentParser();p.add_argument('--assets',required=True);a=p.parse_args()
    root=Path(a.assets)
    w=torch.load(root/'ac_weights_last.pt',map_location='cpu',weights_only=True)
    arrays={k:v.numpy() for k,v in w.items() if k.startswith(('actor_body.','adaptation_module.'))}
    assert arrays['actor_body.0.weight'].shape==(512,2102)
    assert arrays['adaptation_module.0.weight'].shape==(256,2100)
    np.savez(root/'policy.npz',**arrays)
    torch.storage._load_from_bytes=lambda b:torch.load(io.BytesIO(b),map_location='cpu',weights_only=False)
    upstream=pickle.load(open(root/'parameters.pkl','rb'));cfg=upstream['Cfg']
    sections=['env','init_state','control','obs_scales','normalization','sim']
    (root/'policy_config.json').write_text(json.dumps({k:cfg[k] for k in sections},default=lambda x:x.tolist() if hasattr(x,'tolist') else x))
    assert cfg['env']['num_observations']==70 and cfg['env']['num_observation_history']==30
    assert upstream['AC_Args']['activation']=='elu'
    assert cfg['commands']['num_commands']==15
    assert cfg['env']['observe_clock_inputs'] and cfg['env']['observe_two_prev_actions']
    assert not any(cfg['env'][k] for k in ['observe_vel','observe_yaw','observe_contact_states','observe_only_lin_vel'])
    assert cfg['control']['stiffness']['joint']==20 and cfg['control']['damping']['joint']==.5
    from model import NAMES,HOME
    np.testing.assert_allclose([cfg['init_state']['default_joint_angles'][n] for n in NAMES],HOME)
    assert cfg['obs_scales']['dof_pos']==1 and cfg['obs_scales']['dof_vel']==.05
    assert cfg['control']['action_scale']==.25 and cfg['control']['hip_scale_reduction']==.5
    assert cfg['sim']['dt']*cfg['control']['decimation']==.02
    from policy import FrozenPolicy
    frozen=FrozenPolicy(root/'policy.npz')
    h=np.random.default_rng(0).normal(size=(16,2100)).astype(np.float32)
    def net(x,prefix,layers):
        for i in layers:
            x=F.linear(x,w[f'{prefix}.{i}.weight'],w[f'{prefix}.{i}.bias'])
            if i!=layers[-1]:x=F.elu(x)
        return x
    with torch.no_grad():
        th=torch.from_numpy(h);lat=net(th,'adaptation_module',[0,2,4])
        expected=net(torch.cat([th,lat],dim=-1),'actor_body',[0,2,4,6]).numpy()
    actual=np.asarray(jax.jit(frozen)(jp.asarray(h)))
    np.testing.assert_allclose(actual,expected,rtol=2e-4,atol=2e-4)
    report={'torch_jax_max_abs_error':float(np.max(np.abs(actual-expected))),
        'observation_size':70,'history':30,'joint_order':'FL FR RL RR, hip thigh calf',
        'control_dt_s':.02,'conversion_passed':True,
        'actuation':'WTW hardware deployment PD 20/0.5; original Isaac training actuator_net is not reproduced'}
    (root/'conversion_report.json').write_text(json.dumps(report,indent=2));print(json.dumps(report),flush=True)
if __name__=='__main__':main()

### 3.13 `requirements.txt`

ライブラリの固定バージョン。

In [ ]:
%%writefile /content/unitree_go1_multispeed/requirements.txt
jax[cuda12]==0.6.2
mujoco==3.3.7
mujoco-mjx==3.3.7
brax==0.12.5
flax==0.11.2
optax==0.2.5
chex==0.1.90
toolz==1.0.0
orbax-checkpoint==0.11.23
numpy==2.2.6
scipy==1.15.3
imageio==2.37.0
imageio-ffmpeg==0.6.0
Pillow==11.3.0

### 3.14 `assets_manifest.json`

取得元とSHA-256。

In [ ]:
%%writefile /content/unitree_go1_multispeed/assets_manifest.json
{
  "wtw_commit": "0e7236bdc81ce855cbe3d70345a7899452bdeb1c",
  "menagerie_commit": "0059d4335f8156206f63a35662313385f7ad6d74",
  "files": [
    {
      "path": "ac_weights_last.pt",
      "url": "https://raw.githubusercontent.com/Improbable-AI/walk-these-ways/0e7236bdc81ce855cbe3d70345a7899452bdeb1c/runs/gait-conditioned-agility/pretrain-v0/train/025417.456545/checkpoints/ac_weights_last.pt",
      "sha256": "d1542b0b8c7b767753a286f1cbabe5b4dc86e67611fb95844e870b3c61f8869e"
    },
    {
      "path": "parameters.pkl",
      "url": "https://raw.githubusercontent.com/Improbable-AI/walk-these-ways/0e7236bdc81ce855cbe3d70345a7899452bdeb1c/runs/gait-conditioned-agility/pretrain-v0/train/025417.456545/parameters.pkl",
      "sha256": "eeb132fe125fb4a2169315ae4e3f2d1b1c0a7857fdcb48824d6d375a26268e11"
    },
    {
      "path": "WTW_LICENSE",
      "url": "https://raw.githubusercontent.com/Improbable-AI/walk-these-ways/0e7236bdc81ce855cbe3d70345a7899452bdeb1c/LICENSE",
      "sha256": "7dc66e1919dbcc2b49af464ceac3518b230e0e253d57988d418e23f01ef675b9"
    },
    {
      "path": "unitree_go1/CHANGELOG.md",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/CHANGELOG.md",
      "sha256": "0cf56d81d15241511c79e8c5e83dcd069b64df30103e97227207e4dfbf0775dc"
    },
    {
      "path": "unitree_go1/LICENSE",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/LICENSE",
      "sha256": "2b794acf1e250f5545ae5a47d81e5685ea5488ce077d1087ed8be5f81e9ba5c0"
    },
    {
      "path": "unitree_go1/README.md",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/README.md",
      "sha256": "a4b63613124da20b09f9f8c550482d670ab2b784a272a489f94e2288df43f151"
    },
    {
      "path": "unitree_go1/assets/calf.stl",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/assets/calf.stl",
      "sha256": "ad3685e132454fd64e4ca3e5089f0d812fb53958e95c14913a22e3193e9e9ed8"
    },
    {
      "path": "unitree_go1/assets/hip.stl",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/assets/hip.stl",
      "sha256": "b6599b9dd2751ebd87449203223f5683a6c235268b76fcc443e00e0d974a06c0"
    },
    {
      "path": "unitree_go1/assets/thigh.stl",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/assets/thigh.stl",
      "sha256": "bbf7c0a441323088c59276dbec8a32e708a1c61f42d30a6f6775176366ea23f8"
    },
    {
      "path": "unitree_go1/assets/thigh_mirror.stl",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/assets/thigh_mirror.stl",
      "sha256": "1a90b89de47a57cb12a2d703f6b008a6ace8a1890a381c27723296fd0d00b9d8"
    },
    {
      "path": "unitree_go1/assets/trunk.stl",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/assets/trunk.stl",
      "sha256": "051a57995dc6f892fc9bc53cd94e9a894f3dc209fd646e94d55d764150013970"
    },
    {
      "path": "unitree_go1/go1.xml",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/go1.xml",
      "sha256": "d5a7466784c8e72fd174cc7fa5a92754a1a0a6d458a1dec79e7ca0abc9525c4a"
    },
    {
      "path": "unitree_go1/scene.xml",
      "url": "https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/0059d4335f8156206f63a35662313385f7ad6d74/unitree_go1/scene.xml",
      "sha256": "eceb4231bf2a3b0a190296963b38b904ed77dc14f5770bbcdd67fb0be8c49374"
    }
  ]
}

### 3.15 `README.md`

使い方・評価方法・検証範囲。

In [ ]:
%%writefile /content/unitree_go1_multispeed/README.md
# Unitree Go1：複数速度学習と3速度の比較

## 実行方法

`unitree_go1_multispeed.ipynb`をGoogle Colabにアップロードし、L4を選んで「すべてのセルを実行」してください。Drive接続の承認が必要です。ZIPアップロードと以前のpolicy.pklは不要です。

ステップ3は、ファイルごとの説明と`%%writefile`セルに分けています。通常の改行・インデントで内容を確認できます。

出力先は`/content/drive/MyDrive/ship_robot_runs/experiment04_multispeed`です。以前の実験を上書きしません。設定を変更する場合はOUTPUTも別名にしてください。

## 学習と比較の設計

- 元の方策：公開済みWalk These Ways Go1。履歴適応モジュールと歩行方策は固定。
- 学習対象：関節目標角への±0.12radの補正モデル1つ。
- 学習速度：各エピソードで0.10〜0.30m/sから一様に抽選し、エピソード内は一定。
- エピソード終了時：Brax標準の初期状態の再利用ではなく、reset_wrapper.pyで速度・揺れ・開始位置を抽選し直す。
- 観測：元の70次元観測に速度指令が含まれ、補正モデルにもその観測を入力。
- 報酬：そのエピソードの速度指令と甲板相対速度との追従誤差を使う。
- 学習開始位置：甲板のx=−12〜+10mから抽選。20秒エピソードで甲板の複数位置を経験する。
- 学習量：揺れ4段階×300万、合計1200万ステップ。学習seed=700。
- 評価速度：0.10・0.20・0.30m/s。全速度で最終段階の同じ補正モデルを使用。
- 評価条件：学習に使わないseed1000〜1004、開始x=−12m、各約60秒、途中リセットなし。
- 比較：公開方策のみ／公開方策＋複数速度で学習した補正。
- 静止甲板でも同じ比較を行う。FLAT_CONTROL=Falseで省略できるが、標準はTrue。

旧版は2m先の目標付近で減速する課題でした。今回は一定速度指令で歩き続ける課題なので、旧版の補正は読み込まず、新しく学習します。公開歩行方策は引き続き学習済みのものを使い、ゼロから歩行を学ぶ工程はありません。

「速度に依存しないモデル」ではなく、「速度指令に応じて動く、複数速度に対応したモデル」です。速度ごとに別モデルを作りません。速度指令の途中変更や範囲外の速度への対応は今回検証しません。

## 揺れ・モデルの変更

揺れの上限は順に[roll度,pitch度,heave m]で、[2,1.4,.02]、[6,4,.04]、[12,8,.07]、[18,12,.10]です。周期2〜6秒の3つの正弦波を平均し、最初の5秒で滑らかに立ち上げます。上限値まで毎回揺れる意味ではありません。

60秒の高速歩行に合わせ、甲板は40m×4m、評価開始位置はx=−12mへ変更しました。公開方策の実測速度が指令を上回る場合もあるため、0.30m/s×60秒の18mより長い余裕を設けています。横方向1.5m、長手方向19.5mを超えれば逸脱判定です。

甲板は計算用の質量100kgのプラットフォームです。延長で増えたpitch慣性に合わせてpitchの駆動ゲインを増やしています。実際の船体重量・流体・船体慣性を再現したものではありません。中心から離れるほど回転による上下動と加速度が増えます。

左右の目標揺れは同一ですが、反力により実際の揺れに差が出る場合があります。deck_tracking_rmseを確認してください。旧版の6m甲板と本版の数値を直接比較しないでください。

## 何を比較するか

1. 約60秒の完走数、転倒数、逸脱数。
2. 実測平均前進速度と、指令に対する速度RMSE。
3. 足先と甲板の相対運動から計算する滑りの近似指標。
4. 世界座標に対する胴体roll/pitchのRMSと標準偏差。
5. 前進距離、甲板の目標運動への追従誤差。

速度・滑り・姿勢の指標は、立ち上がり5秒を除いて集計します。失敗した試行も終了までの値を保存しますが、短い失敗が小さな平均値で有利にならないよう、集計表の連続値は完走試行のみを使用します。左右差は両方が完走した同じseedのみです。

完走数が違う場合は、連続値の平均だけで優劣を判断しないでください。完走は正確な速度追従を意味しません。速度RMSEを併せて見てください。5seedの比較は探索的な結果であり、全条件での優位性を保証しません。

元の方策のMuJoCo移植には速度追従誤差もあります。補正により静止甲板でも改善する場合、揺れへの適応だけでなく移植誤差の補償も含まれます。静止甲板の対照結果を使って検討してください。静止甲板だけで補正を学習した別の対照モデルまでは用意していないため、揺れ学習そのものの厳密な因果効果を示す実験ではありません。

## 保存ファイル

- `level_01`〜`level_04/policy.pkl`：補正モデル。最終比較ではlevel_04の1つを全速度で使用。
- `level_*/progress.jsonl`：学習ステップと実測時間。
- `heldout_baseline`、`heldout_trained`：揺れる甲板での全速度・全seedの数値と軌跡。
- `final_report.json`：揺れる甲板の全結果と速度別集計。
- `speed_summary.csv`：速度別の比較表。
- `paired_differences.csv`：同じseedでの右−左の差。
- `flat_control_report`：静止甲板での比較表と全結果。
- `speed_comparison.png`：Colabのステップ9で生成する比較グラフ。
- `videos/speed_comparison_fixed.mp4`：3速度×20秒の60秒比較動画。
- `videos/speed_0.10_fixed.mp4`など：速度ごとの20秒比較動画。

動画は各速度の評価の最初の20秒を使います。数値評価は各約60秒です。カメラは速度ごとに世界に固定し、左右で同じです。速度が切り替わるところで視点が切り替わります。動画seedは結果の良し悪しで選ばず1000に固定します。終了した側は最終姿勢で静止し、FROZENを表示します。

## 所要時間と再開

前回の利用者のL4ログでは300万ステップの学習が約24分でした。本版の通し実行は未実測です。1200万ステップ学習、揺れる甲板30試行、静止甲板30試行、動画生成を含むため、3〜5時間程度の実験枠を確保してください。これは本版の実測保証ではありません。最初の学習ログから見積もり直してください。

完了した段階は設定・コード・公開方策・モデルのハッシュが一致すればスキップします。途中で切れた段階は最後の保存パラメータから再開し、その段階の予算を繰り返します。オプティマイザと乱数を含む厳密な再開ではありません。

## 検証範囲

作成時のCPU検証はlocal_validation.jsonに記録します。静止甲板の3速度で約60秒の公開方策再生、速度指令の抽選・観測・保持、短いPPO更新、保存した補正の読み込み、短い評価・動画生成、集計と再実行管理を確認します。短い学習更新はAPI接続確認であり、補正の性能改善を示すものではありません。

L4での1200万ステップ学習の成功・本版の通し時間・全速度での優位性は未確認です。改善しない結果や悪化する結果も保存します。

## 公開元・ライブラリ

Go1と公開方策の互換性検査、ライセンス保存、CUDA12版JAXを使う専用venv、JAXをimportしない動画生成プロセスは前版を引き継ぎます。

- WTW commit: `0e7236bdc81ce855cbe3d70345a7899452bdeb1c`
- Menagerie commit: `0059d4335f8156206f63a35662313385f7ad6d74`
- https://github.com/Improbable-AI/walk-these-ways
- https://github.com/google-deepmind/mujoco_menagerie/tree/main/unitree_go1

Warpの任意バックエンド読込警告が出ても本構成の実行バックエンドはJAXです。NaN/Infは転倒へ置き換えず停止します。専用venvでColab標準CUDA13との混在を避けます。aliveはプロセスの存在を示し、計算の進捗を保証しません。

### 3.16 `local_validation.json`

作成時のCPU検証記録。

In [ ]:
%%writefile /content/unitree_go1_multispeed/local_validation.json
{
  "hardware": "Windows CPU; no Colab L4 full curriculum run",
  "static_deck_public_policy_60s": [
    {
      "seed": 800,
      "speed_m_s": 0.1,
      "requested_seconds": 60.0,
      "simulated_seconds": 59.98685073852539,
      "failed": false,
      "termination_reason": "completed",
      "fall_detected": false,
      "completed": true,
      "success": true,
      "forward_displacement_m": 12.25243255496025,
      "start_x_m": -12.0,
      "mean_forward_speed_m_s": 0.20524251372196248,
      "speed_rmse_m_s": 0.10872568107835855,
      "mean_slip_proxy_m2_s2": 0.012774403174049268,
      "body_roll_rms_deg": 1.0495463581054187,
      "body_pitch_rms_deg": 3.956057208739372,
      "body_roll_std_deg": 0.748806817885094,
      "body_pitch_std_deg": 0.3772268752470048,
      "deck_tracking_rmse": [
        0.00023665351203344516,
        2.9033934844793876e-05,
        0.00032436025465103186
      ],
      "metric_window": "t >= 5 s; before failure only; fallback whole trace if duration < 5 s",
      "metric_window_seconds": 55.02,
      "policy": "WTW pretrained only",
      "config": {
        "roll_deg": 0.0,
        "pitch_deg": 0.0,
        "heave_m": 0.0,
        "speed": 0.1,
        "random_speed": false,
        "random_start": false,
        "speed_min": 0.1,
        "speed_max": 0.3,
        "sim_dt": 0.0025,
        "substeps": 8
      },
      "no_episode_resets": true
    },
    {
      "seed": 800,
      "speed_m_s": 0.2,
      "requested_seconds": 60.0,
      "simulated_seconds": 59.98685073852539,
      "failed": false,
      "termination_reason": "completed",
      "fall_detected": false,
      "completed": true,
      "success": true,
      "forward_displacement_m": 18.780138969421387,
      "start_x_m": -12.0,
      "mean_forward_speed_m_s": 0.3148714387007342,
      "speed_rmse_m_s": 0.1185538662455325,
      "mean_slip_proxy_m2_s2": 0.016590590435158238,
      "body_roll_rms_deg": 1.0770354940728761,
      "body_pitch_rms_deg": 3.4631188545112197,
      "body_roll_std_deg": 0.7377185949883092,
      "body_pitch_std_deg": 0.3401568901035121,
      "deck_tracking_rmse": [
        0.00023659598196662038,
        1.9954573764874312e-05,
        0.0002852891875120346
      ],
      "metric_window": "t >= 5 s; before failure only; fallback whole trace if duration < 5 s",
      "metric_window_seconds": 55.02,
      "policy": "WTW pretrained only",
      "config": {
        "roll_deg": 0.0,
        "pitch_deg": 0.0,
        "heave_m": 0.0,
        "speed": 0.2,
        "random_speed": false,
        "random_start": false,
        "speed_min": 0.1,
        "speed_max": 0.3,
        "sim_dt": 0.0025,
        "substeps": 8
      },
      "no_episode_resets": true
    },
    {
      "seed": 800,
      "speed_m_s": 0.3,
      "requested_seconds": 60.0,
      "simulated_seconds": 59.98685073852539,
      "failed": false,
      "termination_reason": "completed",
      "fall_detected": false,
      "completed": true,
      "success": true,
      "forward_displacement_m": 25.22403335571289,
      "start_x_m": -12.0,
      "mean_forward_speed_m_s": 0.4228616859405009,
      "speed_rmse_m_s": 0.1267377276946457,
      "mean_slip_proxy_m2_s2": 0.022481205564164167,
      "body_roll_rms_deg": 1.1449221127486793,
      "body_pitch_rms_deg": 2.9448842963895756,
      "body_roll_std_deg": 0.7381091312649803,
      "body_pitch_std_deg": 0.3446725485507928,
      "deck_tracking_rmse": [
        0.00023464905289341913,
        6.897161018736247e-05,
        0.00034422935152657845
      ],
      "metric_window": "t >= 5 s; before failure only; fallback whole trace if duration < 5 s",
      "metric_window_seconds": 55.02,
      "policy": "WTW pretrained only",
      "config": {
        "roll_deg": 0.0,
        "pitch_deg": 0.0,
        "heave_m": 0.0,
        "speed": 0.3,
        "random_speed": false,
        "random_start": false,
        "speed_min": 0.1,
        "speed_max": 0.3,
        "sim_dt": 0.0025,
        "substeps": 8
      },
      "no_episode_resets": true
    }
  ],
  "short_residual_restore_6s": [
    {
      "seed": 900,
      "speed_m_s": 0.1,
      "requested_seconds": 6.0,
      "simulated_seconds": 6.000113487243652,
      "failed": false,
      "termination_reason": "completed",
      "fall_detected": false,
      "completed": true,
      "success": true,
      "forward_displacement_m": 1.2306385040283203,
      "start_x_m": -12.0,
      "mean_forward_speed_m_s": 0.25690317709071964,
      "speed_rmse_m_s": 0.16896097165844556,
      "mean_slip_proxy_m2_s2": 0.02619735370285971,
      "body_roll_rms_deg": 2.294494606535039,
      "body_pitch_rms_deg": 2.703963287395641,
      "body_roll_std_deg": 1.809678534112604,
      "body_pitch_std_deg": 0.97401189602996,
      "deck_tracking_rmse": [
        0.0006722739784633558,
        0.0004051196789423798,
        0.0005533530956007266
      ],
      "metric_window": "t >= 5 s; before failure only; fallback whole trace if duration < 5 s",
      "metric_window_seconds": 1.02,
      "policy": "work/ms_reset_fast_tiny/policy.pkl",
      "config": {
        "roll_deg": 2.0,
        "pitch_deg": 1.4,
        "heave_m": 0.02,
        "speed": 0.1,
        "random_speed": false,
        "random_start": false,
        "speed_min": 0.1,
        "speed_max": 0.3,
        "sim_dt": 0.0025,
        "substeps": 8
      },
      "no_episode_resets": true
    },
    {
      "seed": 900,
      "speed_m_s": 0.2,
      "requested_seconds": 6.0,
      "simulated_seconds": 6.000113487243652,
      "failed": false,
      "termination_reason": "completed",
      "fall_detected": false,
      "completed": true,
      "success": true,
      "forward_displacement_m": 1.8400821685791016,
      "start_x_m": -12.0,
      "mean_forward_speed_m_s": 0.36444824612608145,
      "speed_rmse_m_s": 0.17428116476746916,
      "mean_slip_proxy_m2_s2": 0.031286830191603776,
      "body_roll_rms_deg": 2.2957107700067874,
      "body_pitch_rms_deg": 2.2638158468040204,
      "body_roll_std_deg": 1.5739882566084689,
      "body_pitch_std_deg": 1.414516399346175,
      "deck_tracking_rmse": [
        0.0006835834585177804,
        0.00032955815437432325,
        0.000540451876862112
      ],
      "metric_window": "t >= 5 s; before failure only; fallback whole trace if duration < 5 s",
      "metric_window_seconds": 1.02,
      "policy": "work/ms_reset_fast_tiny/policy.pkl",
      "config": {
        "roll_deg": 2.0,
        "pitch_deg": 1.4,
        "heave_m": 0.02,
        "speed": 0.2,
        "random_speed": false,
        "random_start": false,
        "speed_min": 0.1,
        "speed_max": 0.3,
        "sim_dt": 0.0025,
        "substeps": 8
      },
      "no_episode_resets": true
    },
    {
      "seed": 900,
      "speed_m_s": 0.3,
      "requested_seconds": 6.0,
      "simulated_seconds": 6.000113487243652,
      "failed": false,
      "termination_reason": "completed",
      "fall_detected": false,
      "completed": true,
      "success": true,
      "forward_displacement_m": 2.478160858154297,
      "start_x_m": -12.0,
      "mean_forward_speed_m_s": 0.44729984916892707,
      "speed_rmse_m_s": 0.15565189484703687,
      "mean_slip_proxy_m2_s2": 0.02844104466417133,
      "body_roll_rms_deg": 1.7639944987319969,
      "body_pitch_rms_deg": 1.9294371160692012,
      "body_roll_std_deg": 1.5025234716202152,
      "body_pitch_std_deg": 1.5891373820627686,
      "deck_tracking_rmse": [
        0.0006876542168034166,
        0.0003823083332104206,
        0.0005250201650202361
      ],
      "metric_window": "t >= 5 s; before failure only; fallback whole trace if duration < 5 s",
      "metric_window_seconds": 1.02,
      "policy": "work/ms_reset_fast_tiny/policy.pkl",
      "config": {
        "roll_deg": 2.0,
        "pitch_deg": 1.4,
        "heave_m": 0.02,
        "speed": 0.3,
        "random_speed": false,
        "random_start": false,
        "speed_min": 0.1,
        "speed_max": 0.3,
        "sim_dt": 0.0025,
        "substeps": 8
      },
      "no_episode_resets": true
    }
  ],
  "sampling_test": "8 distinct commands in [0.10,0.30], observation/history command matches, held constant within episode, randomized start in range",
  "resampling_wrapper_test": "command resampled at every forced episode boundary; observation agrees; no within-episode resampling",
  "cached_reset_pose_test": "xpos, geom_xpos and xmat agree with a fresh native forward at three randomized reset positions within 2e-6",
  "orchestration_fixture_tests": [
    "single checkpoint for all speeds",
    "completed-only means",
    "jointly completed paired differences",
    "flat controls",
    "completion cache skips"
  ],
  "render_test": {
    "speeds": [
      0.1,
      0.2,
      0.3
    ],
    "seconds_per_speed": 2,
    "total_seconds": 6,
    "visually_checked": true
  },
  "full_training_performance_verified": false,
  "short_ppo_update": {
    "step": 8,
    "elapsed_s": 211.875,
    "eval/walltime": 159.9476854801178,
    "training/sps": 0.15894625287019964,
    "training/walltime": 50.3314790725708,
    "training/entropy_loss": -0.06370401382446289,
    "training/policy_loss": 0.15712693333625793,
    "training/total_loss": 0.4722440242767334,
    "training/v_loss": 0.37882110476493835,
    "eval/episode_height": 2.763993740081787,
    "eval/episode_reward": 8.207679748535156,
    "eval/episode_slip_proxy": 0.9053974151611328,
    "eval/episode_velocity": 0.5937239527702332,
    "eval/episode_height_std": 0.012732982635498047,
    "eval/episode_reward_std": 0.4878535270690918,
    "eval/episode_slip_proxy_std": 0.14650335907936096,
    "eval/episode_velocity_std": 0.045253753662109375,
    "eval/avg_episode_length": 10.0,
    "eval/std_episode_length": 0.0,
    "eval/epoch_eval_time": 0.1967933177947998,
    "eval/sps": 101.62946701703757
  }
}

### 3.17 実験設定とコードの控えを保存

In [ ]:
config_path = WORK / 'experiment.json'
config_path.write_text(json.dumps(CONFIG, indent=2), encoding='utf-8')
source_names = [
    "model.py",
    "policy.py",
    "ship_env.py",
    "reset_wrapper.py",
    "train.py",
    "evaluate_speeds.py",
    "report.py",
    "run_experiment.py",
    "render_speeds.py",
    "fetch_assets.py",
    "convert.py",
    "requirements.txt",
    "assets_manifest.json",
    "README.md",
    "local_validation.json"
]
for name in source_names:
    assert (WORK / name).is_file(), f'未保存: {name}'
    shutil.copy2(WORK / name, archive / name)
shutil.copy2(config_path, archive / config_path.name)
print('コードと実験設定を保存しました')

## 4 専用Python環境を用意
標準Colab環境と分離し、CUDA12版JAXを使います。PyTorchは重み変換用のCPU版です。
ノートブック本体はJAXをimportせず、インストール後の再起動を不要にします。
30秒ごとのalive表示は、プロセスが終了していないことを示します。計算の進捗を保証する表示ではありません。

In [ ]:
VENV=Path('/content/unitree_go1_multispeed_venv');PY=VENV/'bin/python'
ENV=os.environ.copy()
for name in ['PYTHONPATH','PYTHONHOME','JAX_PLATFORMS']:ENV.pop(name,None)
ENV.update(PYTHONNOUSERSITE='1',MUJOCO_GL='egl',XLA_PYTHON_CLIENT_PREALLOCATE='false',IMAGEIO_FFMPEG_NO_PREVENT_SIGINT='1')
if not PY.exists():
    subprocess.run([sys.executable,'-m','venv','--without-pip',str(VENV)],check=True,env=ENV)
stamp=VENV/'requirements.sha256'
signature=hashlib.sha256((WORK/'requirements.txt').read_bytes()+b'torch-cpu-2.7.1').hexdigest()
pip=[sys.executable,'-m','pip','--python',str(PY)]
if not stamp.exists() or stamp.read_text()!=signature:
    subprocess.run(pip+['install','torch==2.7.1','--index-url','https://download.pytorch.org/whl/cpu'],check=True,env=ENV)
    subprocess.run(pip+['install','-r',str(WORK/'requirements.txt')],check=True,env=ENV)
    subprocess.run(pip+['check'],check=True,env=ENV)
    stamp.write_text(signature)
print('環境準備完了:',PY)

def run_child(args,log_name,cpu=False):
    child_env=ENV.copy()
    if cpu:child_env['JAX_PLATFORMS']='cpu'
    p=subprocess.Popen([str(PY),'-u',*map(str,args)],cwd=WORK,env=child_env,
                       stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    messages=queue.Queue()
    def reader():
        for line in p.stdout:messages.put(line)
        messages.put(None)
    threading.Thread(target=reader,daemon=True).start()
    started=time.monotonic()
    try:
        with (Path(OUTPUT)/log_name).open('a',encoding='utf-8') as log:
            while True:
                try:line=messages.get(timeout=30)
                except queue.Empty:line=f'[alive] 経過 {(time.monotonic()-started)/60:.1f} 分（進捗の保証ではありません）\n'
                if line is None:break
                print(line,end='',flush=True);log.write(line);log.flush()
        if p.wait():raise RuntimeError(f'処理が停止しました。直前の出力と {log_name} を確認してください。')
    finally:
        if p.poll() is None:
            os.killpg(p.pid,signal.SIGTERM)
            try:p.wait(timeout=10)
            except subprocess.TimeoutExpired:os.killpg(p.pid,signal.SIGKILL);p.wait()

## 5 GPU・公開方策の互換性を確認
Go1関節の順序、初期角、70次元×履歴30、20ms制御、行動倍率を照合し、PyTorch/JAXの出力を比較します。
公開ファイルは全ハッシュを照合します。検証失敗時は追加学習へ進みません。

In [ ]:
subprocess.run(['nvidia-smi'], check=True)
run_child(['-c', "import jax; print(jax.__version__, jax.devices()); assert jax.default_backend()=='gpu'"], 'gpu_check.log')
run_child(['fetch_assets.py', '--assets', CONFIG['assets']], 'fetch.log', cpu=True)
run_child(['convert.py', '--assets', CONFIG['assets']], 'conversion.log', cpu=True)
print(json.loads((Path(CONFIG['assets']) / 'conversion_report.json').read_text()))

## 6 静止甲板で3速度を確認（学習なし）
公開方策のみで、各速度20秒の再生を確認します。速度指令に対して実測速度も表示します。
全速度で非転倒・非逸脱、有限の指標、前進を確認できなければ停止します。
これは完全な速度一致を要求する検査ではありません。移植による追従誤差も後の比較対象です。

In [ ]:
run_child([
    'evaluate_speeds.py', '--assets', CONFIG['assets'],
    '--out', Path(OUTPUT) / 'compatibility', '--condition', 0, 0, 0,
    '--speeds', *CONFIG['eval_speeds'], '--seeds', 800, '--seconds', 20,
], 'compatibility.log')
checks = json.loads((Path(OUTPUT) / 'compatibility/results.json').read_text())
assert all(r['completed'] and r['forward_displacement_m'] > 0.5 for r in checks), '静止甲板の確認に失敗しました'
for r in checks:
    print(f"指令 {r['speed_m_s']:.2f} m/s | 実測 {r['mean_forward_speed_m_s']:.3f} m/s | RMSE {r['speed_rmse_m_s']:.3f} m/s")

## 7 複数速度で学習し、3速度で比較
速度指令はリセット時に抽選し、方策の履歴観測へ含めます。報酬もその指令への追従誤差を使います。
長い甲板の複数の場所を経験できるよう、学習時の開始x位置は−12〜+10mから抽選します。
20秒の学習エピソードで、速度と揺れの組み合わせを経験します。

評価時は全条件で開始x=−12m、速度一定、同じ5seed、約60秒です。エピソードの途中リセットはありません。
評価seedは条件選択や動画の良い例の探索に使いません。動画は常にseed1000です。
全速度の比較に**最終段階の同じpolicy.pkl**を使います。

完了済み段階は設定・コード・公開方策・保存モデルのハッシュが一致すればスキップします。
中断した段階は最後の保存パラメータから、その段階の予算を再実行します。
オプティマイザ・乱数を含む厳密な再開ではありません。設定を変える場合はOUTPUTを別名にしてください。

In [ ]:
run_child(['run_experiment.py', '--config', WORK / 'experiment.json'], 'experiment_console.log')

## 8 速度別の数値比較
まず完走数・転倒数・逸脱数を見ます。成功は今回は「指定した時間を非転倒・非逸脱で完走」であり、
指定速度への完全一致を意味しません。追従性能は速度RMSEを別に確認します。

速度・滑り・姿勢の集計は開始5秒を除きます。途中終了した試行が低い値で有利にならないよう、
連続値の平均は完走試行のみです。左右差のpaired_differences.csvは両方が完走した同じseedのみを使用します。
完走数が違う場合、平均だけで優劣を判断しないでください。

In [ ]:
import pandas as pd
from IPython.display import display

summary = pd.read_csv(Path(OUTPUT) / 'speed_summary.csv')
display(summary[[
    'speed_m_s', 'controller', 'completed_count', 'count',
    'fall_count', 'out_of_bounds_count', 'mean_forward_speed_m_s',
    'speed_rmse_m_s', 'mean_slip_proxy_m2_s2',
    'body_roll_rms_deg', 'body_pitch_rms_deg',
]].round(4))

print('同じseedの左右差（右−左）:')
pairs = pd.read_csv(Path(OUTPUT) / 'paired_differences.csv')
display(pairs.round(4))

if CONFIG['flat_control']:
    print('静止甲板の対照評価:')
    display(pd.read_csv(Path(OUTPUT) / 'flat_control_report/speed_summary.csv').round(4))

## 9 比較グラフ
完走した試行の平均です。上の完走数と併せて確認してください。欠損は0へ置き換えません。

In [ ]:
import matplotlib.pyplot as plt

metrics = [
    ('speed_rmse_m_s', 'Speed tracking RMSE [m/s]'),
    ('mean_slip_proxy_m2_s2', 'Foot slip proxy [m2/s2]'),
    ('body_roll_rms_deg', 'Body roll RMS [deg]'),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (metric, title) in zip(axes, metrics):
    for label, rows in summary.groupby('controller'):
        rows = rows.sort_values('speed_m_s')
        ax.plot(rows['speed_m_s'], rows[metric], marker='o', label=label)
    ax.set(xlabel='Command speed [m/s]', ylabel=title)
    ax.set_xticks(CONFIG['eval_speeds'])
    ax.grid(alpha=0.3)
axes[0].legend()
fig.suptitle('Completed runs only; compare completion counts first')
fig.tight_layout()
fig.savefig(Path(OUTPUT) / 'speed_comparison.png', dpi=180)
plt.show()

## 10 固定カメラの比較動画をダウンロード
左が公開方策のみ、右が複数速度で学習した補正を追加した方策です。
各速度の最初の20秒を順に並べ、合計60秒にしています。数値評価は各試行約60秒の全区間です。
カメラは速度ごとの区間で世界に固定し、左右は同じ視点です。速度が切り替わるところだけ視点も切り替わります。
転倒・逸脱後は最終姿勢を静止し、FROZENと表示します。

In [ ]:
from google.colab import files
files.download(str(Path(OUTPUT) / 'videos/speed_comparison_fixed.mp4'))

## 検証範囲と解釈
ローカルCPUでの確認内容はREADMEとlocal_validation.jsonに記録しています。
L4での1200万ステップ通し学習、複数速度での改善は未検証です。改善しない・悪化する結果もそのまま保存します。

甲板は40m×4m、質量100kgの計算用プラットフォームです。ロボット・甲板・接触を物理計算しますが、船体の実物寸法や慣性を再現したモデルではありません。
延長で増えるpitch慣性に合わせて駆動ゲインも変更しています。以前の6m甲板と数値を直接比較しないでください。
長い甲板では中心から離れるほど回転による上下動・加速度が増えます。開始位置・揺れの目標・速度は左右で同じです。
deck_tracking_rmseも記録するため、左右の甲板運動の追従誤差を確認できます。

元の公開方策の物理移植には速度追従誤差があり、補正はその誤差と揺れの両方を補償し得ます。
静止甲板の対照評価も見ることで、揺れに特有の改善かを検討してください。ただし同じ方策を静止床だけで学習した対照群との厳密な因果比較ではありません。
学習範囲外の速度や実機への対応を保証するものでもありません。

出典：[Walk These Ways](https://github.com/Improbable-AI/walk-these-ways)、
[MuJoCo Menagerie Go1](https://github.com/google-deepmind/mujoco_menagerie/tree/main/unitree_go1)。
コミット・ハッシュは固定し、公開資産のライセンスも保存します。